# Name Cleaning & Frequency Analysis — v13
## Sondondo Valley Colonial Records

v13 incorporates Prof. review edits from `name_frequency_analysis_v5_jcp.xlsx`:
- **883 gender assignments** for previously-Unknown names (applied as the highest-priority
  gender source, ahead of Chipao and persona_type)
- **75 first-name spelling/typo corrections** + **15 new abbreviation-duplicate patterns**
  (e.g. `manl manuel flores` → `manuel flores`), merged into `name_cleaning_rules.json`
- **11 structural reclassifications** — entries sitting in first_name/middle_name that are
  actually last names (e.g. `huarcaya`, `sanchez`), moved to the last_name category
- **6 special one-off structural fixes** (leaked surname duplicates, a title prefix, a
  no-space full name, and one plain typo pair the phonetic rules didn't catch)

**Input files:**
- `accura_personas.csv`, `chipao_personas.csv`
- `name_cleaning_rules.json` (already updated with Prof.'s spelling fixes)
- `professor_reclassify_targets.json`, `professor_special_overrides.json`,
  `professor_edits_parsed.csv` (Prof.'s structural fixes and gender assignments)

**Output files:**
- `accura_personas_cleaned.csv`, `name_frequency_analysis_v13.csv`,
  `gender_conflicts_review.csv`


## Part A: Name Cleaning

### 1. Import libraries

In [ ]:
import pandas as pd
import re
import json

### 2. Load data, cleaning rules, and Prof.'s corrections

In [ ]:
ACCURA = pd.read_csv("accura_personas_v13_preprocessed.csv", low_memory=False)
CHIPAO = pd.read_csv("chipao_personas.csv", low_memory=False)

with open("name_cleaning_rules.json", "r", encoding="utf-8") as f:
    RULES = json.load(f)
with open("professor_reclassify_targets.json", "r", encoding="utf-8") as f:
    RECLASSIFY = json.load(f)
with open("professor_special_overrides.json", "r", encoding="utf-8") as f:
    SPECIAL_OVERRIDES = json.load(f)

PROF_EDITS = pd.read_csv("professor_edits_parsed.csv")
PROF_EDITS["actual_gender"] = PROF_EDITS["actual_gender"].replace({"non applicable": "not applicable"})

print("accura:", ACCURA.shape)
print("chipao:", CHIPAO.shape)
print("prof edits:", PROF_EDITS.shape)

### 3. Normalize text

In [ ]:
def norm(text):
    """Lowercase, strip, and collapse whitespace. Returns '' for missing values."""
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).strip().lower())

ACCURA["name_norm"] = ACCURA["name"].apply(norm)
ACCURA["lastname_norm"] = ACCURA["lastname"].apply(norm)

### 4. Fix name contamination (surname leaked into name field)

In [ ]:
CONNECTORS = {"de", "del", "la", "los", "las"}

def clean_name_field(name_norm, lastname_norm):
    if not name_norm or not lastname_norm:
        return name_norm
    tokens = name_norm.split(" ")
    last_tokens = set(lastname_norm.split(" "))
    while len(tokens) > 1 and tokens[-1] in last_tokens:
        tokens.pop()
        while len(tokens) > 1 and tokens[-1] in CONNECTORS:
            tokens.pop()
    return " ".join(tokens)

ACCURA["name_decontaminated"] = ACCURA.apply(
    lambda row: clean_name_field(row["name_norm"], row["lastname_norm"]), axis=1
)
print(f"{(ACCURA['name_decontaminated'] != ACCURA['name_norm']).sum():,} rows decontaminated")

### 5. Apply the JSON cleaning rules

This now includes Prof.'s 75 typo corrections, 15 new abbreviation-duplicate patterns,
and a new `title_prefix` rule (v13) that strips leading titles/offices that aren't part of
a name (e.g. "doctor", "presbítero" - a religious office), confirmed by Prof.'s review.

In [ ]:
def apply_name_rules(name_norm_original, name_decontaminated, rules):
    for placeholder in rules["placeholders"]:
        if placeholder in name_norm_original:
            return "__EXCLUDE__"

    tokens = name_decontaminated.split(" ")

    # strip a leading title/office word (e.g. "doctor", "presbítero") - these
    # aren't part of the name, just how the record refers to the person's role
    title_prefix = set(rules.get("title_prefix", []))
    while len(tokens) > 1 and tokens[0] in title_prefix:
        tokens.pop(0)

    abbrevs = rules.get("abbreviations", {})
    if tokens[0] in abbrevs:
        expanded = abbrevs[tokens[0]]
        if len(tokens) > 1 and tokens[1] == expanded:
            tokens = tokens[1:]
        else:
            tokens[0] = expanded

    short_exclude = set(rules.get("short_exclude", []))
    while len(tokens) > 1 and tokens[0] in short_exclude:
        tokens.pop(0)
    if len(tokens) == 1 and tokens[0] in short_exclude:
        return "__EXCLUDE__"

    variants = rules.get("spelling_variants", {})
    tokens = [variants.get(t, t) for t in tokens]

    return " ".join(tokens)

def apply_lastname_rules(lastname_norm, rules):
    name = lastname_norm
    if not name:
        return name
    for placeholder in rules["placeholders"]:
        if placeholder in name:
            return "__EXCLUDE__"
    short_exclude = set(rules.get("short_exclude", []))
    if name in short_exclude:
        return "__EXCLUDE__"
    variants = rules.get("spelling_variants", {})
    tokens = name.split(" ")
    tokens = [variants.get(t, t) for t in tokens]
    return " ".join(tokens)

In [ ]:
ACCURA["fixed_name"] = ACCURA.apply(
    lambda row: apply_name_rules(row["name_norm"], row["name_decontaminated"], RULES["name_rules"]),
    axis=1
)
ACCURA["fixed_lastname"] = ACCURA["lastname_norm"].apply(
    lambda n: apply_lastname_rules(n, RULES["lastname_rules"])
)
print(f"name excluded: {(ACCURA['fixed_name']=='__EXCLUDE__').sum():,}")
print(f"lastname excluded: {(ACCURA['fixed_lastname']=='__EXCLUDE__').sum():,}")

### 6. Extract standardized first name

In [ ]:
ACCURA["first_name_std"] = ACCURA["fixed_name"].apply(
    lambda s: norm(s).split(" ")[0] if pd.notna(s) and s != "__EXCLUDE__" else ""
)

### 7. Split first_name / middle_name, build full_name

In [ ]:
def split_first_middle(name):
    if not name or name == "__EXCLUDE__":
        return ("", "")
    tokens = norm(name).split(" ")
    return (tokens[0], " ".join(tokens[1:]))

split_result = ACCURA["fixed_name"].apply(split_first_middle)
ACCURA["first_name"] = split_result.apply(lambda t: t[0])
ACCURA["middle_name"] = split_result.apply(lambda t: t[1])

ACCURA["full_name_clean"] = ACCURA.apply(
    lambda row: (norm(row["fixed_name"]) + " " + norm(row["fixed_lastname"])).strip()
    if row["fixed_name"] != "__EXCLUDE__" and row["fixed_lastname"] != "__EXCLUDE__"
    else "__EXCLUDE__", axis=1
)
print(f"personas with a middle name: {(ACCURA['middle_name'] != '').sum():,}")

---
## Part B: Apply Prof.'s Structural Fixes

Two kinds of structural corrections from the review:
1. **Reclassify as last name** — 11 values sitting in first_name/middle_name that are
   actually surnames. Moved into `fixed_lastname`, cleared from their old slot.
2. **Special one-off overrides** — 6 specific records where the name got split wrong for
   reasons that don't fit a general rule (a leaked duplicate, a title prefix, a no-space
   full name, one uncaught typo pair). Matched by their exact original full name string
   and corrected directly.

In [ ]:
# --- 1. reclassify-as-lastname: first_name-level targets ---
for target in RECLASSIFY["first_name"]:
    mask = ACCURA["first_name"] == target
    n = mask.sum()
    if n:
        # move into lastname (prepend to any existing lastname), clear first_name/middle_name
        ACCURA.loc[mask, "fixed_lastname"] = (
            target + " " + ACCURA.loc[mask, "fixed_lastname"].replace("__EXCLUDE__", "")
        ).str.strip()
        ACCURA.loc[mask, "first_name"] = ""
        ACCURA.loc[mask, "fixed_name"] = "__EXCLUDE__"
        print(f"reclassified {n} rows: first_name '{target}' -> lastname")

In [ ]:
# --- 1b. reclassify-as-lastname: middle_name-level targets ---
for target in RECLASSIFY["middle_name"]:
    mask = ACCURA["middle_name"] == target
    n = mask.sum()
    if n:
        # "gua amani" also needs its own spelling fixed to "guamani" while reclassifying
        clean_target = "guamani" if target == "gua amani" else target
        ACCURA.loc[mask, "fixed_lastname"] = (
            clean_target + " " + ACCURA.loc[mask, "fixed_lastname"].replace("__EXCLUDE__", "")
        ).str.strip()
        ACCURA.loc[mask, "middle_name"] = ""
        print(f"reclassified {n} rows: middle_name '{target}' -> lastname ('{clean_target}')")

In [ ]:
# --- 2. special one-off overrides, matched by original full_name_clean ---
for orig_string, fix in SPECIAL_OVERRIDES.items():
    mask = ACCURA["full_name_clean"] == orig_string
    n = mask.sum()
    if n == 0:
        continue
    if fix.get("exclude_first"):
        ACCURA.loc[mask, "first_name"] = ""
        ACCURA.loc[mask, "fixed_name"] = "__EXCLUDE__"
    else:
        ACCURA.loc[mask, "first_name"] = fix["first"]
    ACCURA.loc[mask, "middle_name"] = fix.get("middle", "")
    if "lastname_override" in fix:
        ACCURA.loc[mask, "fixed_lastname"] = fix["lastname_override"]
    print(f"special override applied to {n} row(s): '{orig_string}'")

In [ ]:
# rebuild full_name_clean after all structural fixes above
ACCURA["full_name_clean"] = ACCURA.apply(
    lambda row: (row["first_name"] + (" " + row["middle_name"] if row["middle_name"] else "")
                 + " " + row["fixed_lastname"]).strip()
    if row["first_name"] and row["fixed_lastname"] not in ("", "__EXCLUDE__")
    else "__EXCLUDE__", axis=1
)

# IMPORTANT: also refresh first_name_std, since the structural fixes above may have
# changed first_name (e.g. 'huarcaya' -> 'gregorio' for the leaked-surname-prefix case).
# Gender resolution below matches on first_name_std, so this must reflect the corrected value.
ACCURA["first_name_std"] = ACCURA["first_name"]

---
## Part B.5: Apply Prof.'s Expert-Vetted 284-Case Review (v13)

Prof. manually reviewed all 284 records with a 3+ part given name, checking original
document images where needed. His corrections use a 4-column structure: name /
middle_name_or_initial / paternal_last_name / maternal_last_name - plus sex for every row.
This is applied as the final, authoritative layer for these specific records, overriding
whatever the automated rules above produced for them.

Matched on (event_idno, persona_type), which is unique across the review set (name-string
matching was tried in earlier rounds and proved fragile once spelling corrections changed
the very strings being matched on - this avoids that class of bug entirely).

New in v13: a `maternal_last_name` column (8 of the 287 reviewed records have a genuine
second surname identified; blank for everyone else, since our data doesn't otherwise
distinguish paternal from maternal surnames).

In [ ]:
PROF_284 = pd.read_csv("professor_284_final_overrides.csv")
print(f"reviewed records to apply: {len(PROF_284)}")

ACCURA["maternal_last_name"] = ""

SEX_MAP = {"m": "Male", "f": "Female"}
name_variants = RULES["name_rules"]["spelling_variants"]
lastname_variants = RULES["lastname_rules"]["spelling_variants"]

def std_name_field(value, variants):
    """Apply the same spelling-variant standardization used everywhere else in the
    pipeline, so Prof.'s overrides stay consistent (e.g. 'guamani' -> 'huamani')."""
    if pd.isna(value) or value == "":
        return value
    tokens = norm(value).split(" ")
    return " ".join(variants.get(t, t) for t in tokens)

applied, skipped = 0, []

for _, row in PROF_284.iterrows():
    # try the simple match first: (event_idno, persona_type) alone
    mask = (ACCURA["event_idno"] == row["event_idno"]) & (ACCURA["persona_type"] == row["persona_type"])
    n = mask.sum()

    if n > 1:
        # ambiguous (e.g. multiple witnesses/godparents at the same event, or twins
        # sharing persona_type='baptized') - narrow down using name + lastname too
        mask2 = mask & (ACCURA["name"] == row["current_name"]) & (ACCURA["lastname"] == row["current_lastname"])
        if mask2.sum() == 1:
            mask = mask2
            n = 1

    if n != 1:
        skipped.append((row["event_idno"], row["persona_type"], row["current_name"], n))
        continue

    ACCURA.loc[mask, "first_name"] = std_name_field(row["name"], name_variants)
    ACCURA.loc[mask, "middle_name"] = std_name_field(row["middle_name_or_initial"], name_variants) if pd.notna(row["middle_name_or_initial"]) else ""
    if pd.notna(row["paternal_last_name"]):
        ACCURA.loc[mask, "fixed_lastname"] = std_name_field(row["paternal_last_name"], lastname_variants)
    ACCURA.loc[mask, "maternal_last_name"] = std_name_field(row["maternal_last_name"], lastname_variants) if pd.notna(row["maternal_last_name"]) else ""
    ACCURA.loc[mask, "first_name_std"] = ACCURA.loc[mask, "first_name"]
    ACCURA.loc[mask, "gender_final_284review"] = SEX_MAP.get(str(row["sex"]).strip().lower(), "Unknown")
    applied += 1

print(f"applied: {applied}")
print(f"skipped (no unique match found): {len(skipped)}")
for s in skipped:
    print("  ", s)

In [ ]:
# rebuild full_name_clean once more, now including maternal_last_name where present
def build_full_name(row):
    if row["first_name"] in ("", "__EXCLUDE__") or row["fixed_lastname"] in ("", "__EXCLUDE__"):
        return "__EXCLUDE__"
    parts = [row["first_name"]]
    if row["middle_name"]:
        parts.append(str(row["middle_name"]))
    parts.append(row["fixed_lastname"])
    if row["maternal_last_name"]:
        parts.append(str(row["maternal_last_name"]))
    return " ".join(parts)

ACCURA["full_name_clean"] = ACCURA.apply(build_full_name, axis=1)

### Save the cleaned dataset

In [ ]:
ACCURA.to_csv("accura_personas_cleaned.csv", index=False, encoding="utf-8-sig")
print(f"accura_personas_cleaned.csv: {len(ACCURA):,} rows")

---
## Part C: Gender Resolution

**Five stages, in priority order (v13 adds Stage -1):**
0. **Prof. review** (new, highest priority) — 883 manually-assigned genders from the
   spreadsheet review. Matched at the most specific available level: full_name first, then
   first_name, then middle_name.
1. Chipao pair match (first_name + last_name exact match)
2. Chipao first-name fallback (>=90% consistency threshold)
3. Persona type fallback (husband/wife/father/mother/etc.)

In [ ]:
# --- build Prof. gender override lookups (excluding the 'not applicable' structural rows) ---
gender_edits = PROF_EDITS[
    (PROF_EDITS["actual_gender"].notna()) & (PROF_EDITS["actual_gender"] != "not applicable")
].copy()
gender_edits["gender_clean"] = gender_edits["actual_gender"].str.capitalize()

# apply the SAME spelling corrections used elsewhere, so the override keys match
# the standardized fixed_name/full_name_clean values used in the rest of the pipeline
fn_variants = RULES["name_rules"]["spelling_variants"]

def std_first(name):
    tok = norm(name).split(" ")[0]
    return fn_variants.get(tok, tok)

full_name_gender = dict(zip(
    gender_edits[gender_edits["name_type"] == "full_name"]["name"].apply(norm),
    gender_edits[gender_edits["name_type"] == "full_name"]["gender_clean"]
))
first_name_gender = dict(zip(
    gender_edits[gender_edits["name_type"] == "first_name"]["name"].apply(std_first),
    gender_edits[gender_edits["name_type"] == "first_name"]["gender_clean"]
))
middle_name_gender = dict(zip(
    gender_edits[gender_edits["name_type"] == "middle_name"]["name"].apply(norm),
    gender_edits[gender_edits["name_type"] == "middle_name"]["gender_clean"]
))

print(f"full_name overrides: {len(full_name_gender)}")
print(f"first_name overrides: {len(first_name_gender)}")
print(f"middle_name overrides: {len(middle_name_gender)}")

In [ ]:
GENDER_MAP = {
    "male": "Male", "mostly_male": "Male",
    "female": "Female", "mostly_female": "Female",
    "unknown": "Unknown", "andy": "Unknown",
}
ACCURA["gender_clean"] = ACCURA["gender"].map(GENDER_MAP)

CHIPAO["fn_norm"] = CHIPAO["First Name"].apply(norm)
CHIPAO["ln_norm"] = CHIPAO["Last Name"].apply(norm)
CHIPAO["sex_norm"] = CHIPAO["Sex"].apply(norm)
chipao_known = CHIPAO[CHIPAO["sex_norm"].isin(["male", "female"])]

pair_lookup = chipao_known.groupby(["fn_norm", "ln_norm"])["sex_norm"].apply(set).to_dict()

fn_gender_counts = chipao_known.groupby(["fn_norm", "sex_norm"]).size().unstack(fill_value=0)
fn_gender_counts.columns = ["female", "male"]
fn_gender_counts["total"] = fn_gender_counts.sum(axis=1)
fn_gender_counts["majority_gender"] = fn_gender_counts.apply(
    lambda r: "Female" if r["female"] >= r["male"] else "Male", axis=1
)
fn_gender_counts["majority_pct"] = fn_gender_counts.apply(
    lambda r: max(r["female"], r["male"]) / r["total"] * 100, axis=1
)

PERSONA_GENDER_MAP = {
    "husband": "Male", "wife": "Female", "father": "Male", "mother": "Female",
    "godfather": "Male", "godmother": "Female",
    "father_of_wife": "Male", "father_of_husband": "Male",
    "mother_of_wife": "Female", "mother_of_husband": "Female",
}
GENDER_THRESHOLD = 90

### Gender resolution (v13: reverted to stages -1 through 3 only)

The internal first-name fallback stage (added in v10) has been removed at the
professor's request - names that can't be resolved through the professor's own
corrections, Chipao, or persona type are left as Unknown rather than inferred from
the dataset's own internal name/gender patterns. The frequency categorization fix
(comparing Unknown-gender names against the full dataset instead of just other
Unknown names) is unrelated to this and stays in place.

In [ ]:
def resolve_gender(row):
    # stage -1 (v8): the 284-case expert-vetted review, checked against original images.
    reviewed = row.get("gender_final_284review")
    if pd.notna(reviewed) and reviewed != "":
        return pd.Series([reviewed, "professor_284_review", False])

    gender = row["gender_clean"]
    if gender != "Unknown":
        return pd.Series([gender, "original", False])

    # stage 0: professor review - full_name is most specific, then first_name, then middle_name
    full_key = row["full_name_clean"]
    if full_key != "__EXCLUDE__" and full_key in full_name_gender:
        return pd.Series([full_name_gender[full_key], "professor_review", False])

    first_std = row["first_name_std"]
    if first_std in first_name_gender:
        return pd.Series([first_name_gender[first_std], "professor_review", False])

    mid_key = row["middle_name"]
    if mid_key and mid_key in middle_name_gender:
        return pd.Series([middle_name_gender[mid_key], "professor_review", False])

    # stage 1: chipao pair match
    pair_key = (first_std, row["lastname_norm"])
    sexes = pair_lookup.get(pair_key)
    if sexes and len(sexes) == 1:
        return pd.Series([next(iter(sexes)).capitalize(), "chipao_pair_match", False])

    # stage 2: chipao first-name fallback
    if first_std and first_std in fn_gender_counts.index:
        row_data = fn_gender_counts.loc[first_std]
        if row_data["majority_pct"] >= GENDER_THRESHOLD:
            return pd.Series([row_data["majority_gender"], "chipao_firstname_fallback", False])

    # stage 3: persona type
    persona_gender = PERSONA_GENDER_MAP.get(row["persona_type"])
    if persona_gender:
        return pd.Series([persona_gender, "persona_type_fallback", False])

    return pd.Series(["Unknown", "unresolved", False])

ACCURA[["gender_final", "gender_source", "gender_conflict"]] = ACCURA.apply(resolve_gender, axis=1)
ACCURA["gender_note"] = ""

print("=== gender resolution complete ===")
print(ACCURA["gender_source"].value_counts())

In [ ]:
gender_conflicts = ACCURA.loc[
    ACCURA["gender_conflict"], ["event_idno", "persona_type", "name", "lastname"]
].reset_index(drop=True)
print(f"gender conflicts: {len(gender_conflicts)}")

In [ ]:
# drop gender_note - it's an internal working column left over from earlier gender
# resolution stages, never populated now that the internal fallback stage is removed
ACCURA = ACCURA.drop(columns=["gender_note"])

# re-save with final gender info
ACCURA.to_csv("accura_personas_cleaned.csv", index=False, encoding="utf-8-sig")
print(f"accura_personas_cleaned.csv (updated): {len(ACCURA):,} rows")

---
## Part D: Name Frequency Analysis

### 8. Build long-format table and aggregate frequency

In [ ]:
def build_type_df(name_col, name_type, use_gender=True):
    d = ACCURA[[name_col, "gender_final", "event_idno"]].rename(columns={name_col: "name"})
    d = d[(d["name"] != "") & (d["name"] != "__EXCLUDE__")]
    d["name_type"] = name_type
    d["gender"] = ACCURA.loc[d.index, "gender_final"] if use_gender else "Not Applicable"
    return d[["name_type", "name", "gender", "event_idno"]]

long_df = pd.concat([
    build_type_df("first_name", "first_name"),
    build_type_df("middle_name", "middle_name"),
    build_type_df("fixed_lastname", "last_name", use_gender=False),
    build_type_df("full_name_clean", "full_name"),
], ignore_index=True)

long_df["name_type"].value_counts()

In [ ]:
name_freq = (
    long_df.groupby(["name_type", "name", "gender"])
    .agg(freq=("event_idno", "size"), event_id=("event_idno", lambda s: "|".join(s)))
    .reset_index()
)
name_freq.sort_values("freq", ascending=False).head(10)

### 9. Categorize by frequency

- **Most common** — top 15%  ·  **Common** — next 35%  ·  **Rare** — next 35%  ·  **Very rare** — bottom 15%

In [ ]:
def categorize_group(freq_df):
    """Assigns Most common / Common / Rare / Very rare based on cumulative
    share of names at each frequency value (ties always share a bucket)."""
    freq_df = freq_df.sort_values("freq", ascending=False).reset_index(drop=True)
    n_names = len(freq_df)
    counts_per_freq = freq_df.groupby("freq").size().sort_index(ascending=False)
    cum_pct = counts_per_freq.cumsum() / n_names * 100

    def bucket(pct):
        if pct <= 15: return "Most common"
        elif pct <= 50: return "Common"
        elif pct <= 85: return "Rare"
        else: return "Very rare"

    freq_to_bucket = {f: bucket(p) for f, p in cum_pct.items()}
    freq_df["categorize"] = freq_df["freq"].map(freq_to_bucket)
    return freq_df

# Male, Female, and Not Applicable (last_name) keep the original per-group logic -
# these are real, distinct populations, and mixing them would hide genuine
# differences (e.g. a name common among men but rare among women).
categorized_parts = []
for (ntype, gender), sub in name_freq.groupby(["name_type", "gender"]):
    if gender == "Unknown":
        continue  # handled separately below
    sub = categorize_group(sub.copy())
    sub["name_type"] = ntype
    sub["gender"] = gender
    categorized_parts.append(sub)

name_frequency_v13 = pd.concat(categorized_parts, ignore_index=True)
print("Male/Female/Not Applicable categorized (unchanged logic)")

### Categorize "Unknown" rows differently (v13 fix)

Unknown isn't a real population the way Male and Female are - it's just "we don't know
this person's gender," a data gap. Ranking Unknown-labeled names only against *other*
Unknown-labeled names was misleading: that pool is tiny (~24 names for first_name), so a
name with freq=2 could look "Common" there while being genuinely rare in the dataset as a
whole.

Fix: for Unknown rows, categorize using the name's TOTAL frequency (Male + Female + Unknown
combined) compared against every other name's total - i.e. "how common is this name
overall" - not "how common is this name among other Unknown-gender entries."

In [ ]:
# build each name's TOTAL frequency across all genders (the real population size)
name_totals = (
    long_df.groupby(["name_type", "name"])
    .size()
    .reset_index(name="total_freq")
)

# bucket names by their overall total, separately per name_type
total_categorized = []
for ntype, sub in name_totals.groupby("name_type"):
    sub = sub.rename(columns={"total_freq": "freq"})
    sub = categorize_group(sub.copy())
    sub = sub.rename(columns={"freq": "total_freq", "categorize": "overall_categorize"})
    sub["name_type"] = ntype
    total_categorized.append(sub)
name_totals_categorized = pd.concat(total_categorized, ignore_index=True)

# apply the OVERALL category to Unknown-gender rows specifically
unknown_rows = name_freq[name_freq["gender"] == "Unknown"].copy()
unknown_rows = unknown_rows.merge(
    name_totals_categorized[["name_type", "name", "overall_categorize"]],
    on=["name_type", "name"], how="left"
)
unknown_rows["categorize"] = unknown_rows["overall_categorize"]
unknown_rows = unknown_rows.drop(columns=["overall_categorize"])

name_frequency_v13 = pd.concat([name_frequency_v13, unknown_rows], ignore_index=True)
name_frequency_v13 = name_frequency_v13[["name_type", "name", "gender", "freq", "categorize", "event_id"]]

print("Unknown rows re-categorized using overall (all-gender) frequency")
name_frequency_v13.groupby(["name_type", "categorize"]).size()

### 10. Cap very long event_id lists

Keeps the file safe to open in Excel (32,767-char cell limit).

In [ ]:
EVENT_ID_CAP = 1500

def cap_event_ids(event_id_str):
    ids = event_id_str.split("|")
    if len(ids) <= EVENT_ID_CAP:
        return event_id_str
    kept = ids[:EVENT_ID_CAP]
    omitted = len(ids) - EVENT_ID_CAP
    return "|".join(kept) + f"|...and {omitted:,} more (total {len(ids):,})"

name_frequency_v13["event_id"] = name_frequency_v13["event_id"].apply(cap_event_ids)
print(f"rows capped: {name_frequency_v13['event_id'].str.contains('more \\(total', na=False).sum()}")

### 11. Save outputs

In [ ]:
name_frequency_v13.to_csv("name_frequency_analysis_v13.csv", index=False, encoding="utf-8-sig")
gender_conflicts.to_csv("gender_conflicts_review.csv", index=False, encoding="utf-8-sig")

print(f"name_frequency_analysis_v13.csv: {len(name_frequency_v13):,} rows")
print(f"gender_conflicts_review.csv: {len(gender_conflicts):,} rows")